## NLP2 Terminology MT – starter

This notebook is a minimal template to:

1. **Run and generate** with a Gemma-style instruction-tuned model.
2. **Evaluate** system outputs with the WMT25 Terminology Shared Task tools in `wmt25-terminology`.
3. **Fine-tune** the model using **Hugging Face Transformers + PEFT**, and **preference optimisation** with **TRL (e.g. DPO)**.

It is designed to run both **locally** (your Jupyter server via `gpu_server.sh`) and on **Google Colab** with Python 3.11. You will need a GPU with enough memory (for 2–4B models, 4-bit loading is recommended).

In [ ]:
# Install dependencies (run once per environment)
import sys, subprocess

packages = [
    "transformers",
    "accelerate",
    "peft",
    "trl",
    "datasets",
    "bitsandbytes",
    "sentencepiece",
    "huggingface_hub",
]

def pip_install(pkgs):
    print("Installing:", pkgs)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *pkgs])

pip_install(packages)

print("Done. You may need to restart the kernel once.")

In [ ]:
# Load a Gemma-style instruct model and generate
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Change this to the exact Gemma variant you want, e.g. "google/gemma-2-4b-it" or a local path.
model_id = "google/gemma-2-2b-it"  # example; replace with e.g. "gemma-3-4b-it" when available

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

generate = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

prompt = (
    "Translate the following sentence to Chinese, respecting the given terminology.\n\n"
    "Source: The patient was diagnosed with chronic kidney disease.\n"
    "Terminology: chronic kidney disease → 慢性肾脏病\n\n"
    "Translation:"
)

out = generate(prompt, max_new_tokens=128, do_sample=True, temperature=0.7)[0]["generated_text"]
print(out)

In [ ]:
# Use WMT25 terminology evaluation scripts
# Assumes this notebook is run from the repo root `nlp2-26/` and that
# `wmt25-terminology/` is present as a sibling directory.

import sys
import subprocess

# Track 2 evaluation (document-level), over all submissions in `ranking/submissions/track2`.
# To evaluate YOUR system, create a new team folder under
#   wmt25-terminology/ranking/submissions/track2/YOURTEAM
# and follow the same file naming scheme as in the examples.

subprocess.run(
    [sys.executable, "ranking/metric_track2/evaluate_track2.py"],
    cwd="wmt25-terminology",
    check=True,
)

In [ ]:
# Supervised fine-tuning (SFT) with Transformers + PEFT (LoRA)

from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
import torch

base_model = model_id  # reuse the model defined above

# Expect JSONL with fields like: {"src": ..., "terms": ..., "tgt": ...}
# Adjust paths/field names to your project.
raw_ds = load_dataset(
    "json",
    data_files={
        "train": "data/train.jsonl",  # TODO: point to your training data
        "eval": "data/dev.jsonl",     # TODO: point to your dev data
    },
)

def format_example(ex):
    src = ex["src"]
    terms = ex.get("terms", "")
    tgt = ex["tgt"]
    prompt = f"Source: {src}\nTerminology: {terms}\nTranslation:"
    return {"text": prompt + " " + tgt}

train_ds = raw_ds["train"].map(format_example, remove_columns=raw_ds["train"].column_names)
val_ds = raw_ds["eval"].map(format_example, remove_columns=raw_ds["eval"].column_names)

tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model_sft = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # may need tweaking per architecture
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_sft = get_peft_model(model_sft, peft_config)

training_config = SFTConfig(
    output_dir="checkpoints/terminology-sft",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    max_seq_length=512,
    logging_steps=10,
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    report_to="none",
)

trainer = SFTTrainer(
    model=model_sft,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_config,
    dataset_text_field="text",
)

trainer.train()

# After training, you can save and later load only the LoRA adapters.
trainer.model.save_pretrained("checkpoints/terminology-sft-lora")

In [ ]:
# Preference optimisation with TRL (DPO-style example)

from datasets import Dataset
from trl import DPOTrainer, DPOConfig
from peft import get_peft_model
from transformers import AutoModelForCausalLM
import torch

# Tiny toy dataset; replace with real preference data where
# each example has: prompt, chosen (better output), rejected (worse output).
po_examples = [
    {
        "prompt": "Translate to Chinese using the given terminology.",
        "chosen": "Better translation that respects the given terms.",
        "rejected": "Worse translation that ignores or mistranslates the terms.",
    }
]

po_ds = Dataset.from_list(po_examples)

po_model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Reuse the same LoRA config as above
po_model = get_peft_model(po_model, peft_config)

dpo_config = DPOConfig(
    output_dir="checkpoints/terminology-dpo",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1,
    max_length=512,
)

dpo_trainer = DPOTrainer(
    model=po_model,
    ref_model=None,  # TRL will create a frozen reference copy by default
    args=dpo_config,
    beta=0.1,
    train_dataset=po_ds,
    tokenizer=tokenizer,
)

dpo_trainer.train()

dpo_trainer.model.save_pretrained("checkpoints/terminology-dpo-lora")